In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from hepattn.experiments.clic.performance.performance import Performance, PerformanceConfig
from hepattn.experiments.clic.performance.plot_helper import PlotHelper

In [ ]:
plt.rcdefaults()
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "serif"
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"
plt.rcParams["legend.fontsize"] = "9"

In [ ]:
# WIS paths (upstream default; kept for reference)
config_dict = {
    "truth_path": "/storage/agrp/dmitrykl/hgpf/hepformer/data/nilo/test_clic_common_raw.root",
    "networks": [
        {
            "name": "mpflow",
            "path": "/storage/agrp/dmitrykl/hgpf/hepattn/src/hepattn/experiments/clic/logs/CLIC_Pflow_FullDiceFocFix_20250613-T142512/ckpts/epoch=159-val_loss=3.51694__test__common.root",
            "network_type": "mpflow_proxy",
            "ind_threshold": 0.65,
        },
        {
            "name": "hgpflow",
            "path": "/storage/agrp/nilotpal/HGPFlow_v2/experiments/hgpflow_v2/clicxminixbig1x2xs2xxxds7kirm1yo/inference/ee_qq_new/pred_test_p8_ee_qq_ecm380_20k_seg_bw0.3_nocut_merged.root",
            "network_type": "hgpflow_proxy",
            "ind_threshold": 0.65,
        },
        {
            "name": "mlpf",
            "path": "/srv01/agrp/dmitrykl/projects/mlpf/particleflow/experiments/pyg-clic-my_20250726_155449_087474/test/preds_common_checkpoint-21-2.657597/batch_size",
            "network_type": "mlpf",
        },
    ],
}

### HiPerGator: paper-tag reproduction

Three trainings of the paper's small model (`configs/base_small.yaml`, 819,683 parameters),
all evaluated with `submit_eval_l4.sh` and read with the **`mpflow_proxy`** convention, which
is the only one comparable to the paper:

| name | code / environment | hardware, matcher | best val_loss |
|---|---|---|---|
| `glow_paper_ref` | paper tag, torch 2.7.0+cu126, scipy matcher | 1 node x 3 L4 | 4.3716 (ep 196) |
| `glow_l4_lap1015` | `clic-paper-main`, torch 2.9.1+cu128 | 1 node x 3 L4, host lap1015_late | 4.2935 (ep 199) |
| `glow_b200_jv` | `clic-paper-main`, torch 2.9.1+cu128 | 1x B200, GPU JV | 4.4089 (ep 199) |

`glow_paper_ref` is the reference: same model, same hardware as `glow_l4_lap1015`, trained before
the ports. The B200 run uses a different batch geometry (2048 vs a global 1020), so its val_loss is
not on the same scale as the other two -- compare it on the plots, not on the loss.

Pandora is added automatically from the raw file, so it appears in every plot without being listed.

In [ ]:
# HiPerGator paths -- SwiGLU/SiLU study: the SiLU arm against the SwiGLU reference at the same
# geometry (B200, JV) and the paper-tag reference.
CLIC_PAPER = "/home/m.mazza/blue/projects/fastml/hepattn-clic-paper/src/hepattn/experiments/clic"
PAPER_MAIN = "/blue/avery/m.mazza/projects/fastml/hepattn-paper/src/hepattn/experiments/clic"

config_dict = {
    "truth_path": "/blue/avery/m.mazza/projects/fastml/hepattn/data/clic/test_clic_common_raw.root",
    "networks": [
        {
            "name": "glow_paper_ref",
            "path": f"{CLIC_PAPER}/logs/clic_paper_small_20260811-T212339/ckpts/epoch=196-val_loss=4.37156__test.root",
            "network_type": "mpflow_proxy",
            "ind_threshold": 0.65,
        },
        {
            "name": "glow_b200_jv",
            "path": f"{PAPER_MAIN}/logs/clic_paper_small_b200_jv_20260911-T141520/ckpts/epoch=199-val_loss=4.40887__test.root",
            "network_type": "mpflow_proxy",
            "ind_threshold": 0.65,
        },
        {
            "name": "glow_silu",
            "path": f"{PAPER_MAIN}/logs/clic_paper_small_silu_20260913-T175916/ckpts/epoch=198-val_loss=4.52969__test.root",
            "network_type": "mpflow_proxy",
            "ind_threshold": 0.65,
        },
    ],
}


In [ ]:
config = PerformanceConfig.from_dict(config_dict)

In [ ]:
perf_obj = Performance(config)

In [ ]:
perf_obj.reorder_and_find_intersection()

In [ ]:
# n_procs: one per allocated CPU; a login/VS Code session usually has 4.
perf_obj.compute_jets(n_procs=4)

In [ ]:
perf_obj.hung_match_jets()
perf_obj.compute_event_features()
perf_obj.compute_jet_res_features(dr_cut=0.1, leading_n_jets=2, pt_min=10)

In [ ]:
from collections import defaultdict


def default_style_dict():
    return {
        "histtype": "step",
        "linewidth": 1,
    }


style_dict = defaultdict(default_style_dict)
# Pandora is the benchmark: filled and in the background.
style_dict["pandora"] = {
    "color": "#003f5c",
    "alpha": 0.3,
    "histtype": "stepfilled",
}
style_dict["glow_paper_ref"] = style_dict["glow_paper_ref"] | {"color": "#ffa600"}
style_dict["glow_silu"] = style_dict["glow_silu"] | {"color": "#ef5675"}
style_dict["glow_b200_jv"] = style_dict["glow_b200_jv"] | {"color": "#7a5195"}

labels = {
    "pandora": "Pandora",
    "glow_paper_ref": "Glow paper tag (3x L4, scipy)",
    "glow_silu": "SiLU feed-forwards (B200, JV)",
    "glow_b200_jv": "SwiGLU reference (B200, JV)",
}

In [ ]:
# Figures belong to the study that asked for them, not to this directory.
STUDY = "/blue/avery/m.mazza/projects/fastml/hepattn-paper/src/hepattn/experiments/clic/studies/swiglu_silu"

plot_helper = PlotHelper(perf_obj, style_dict=style_dict, plot_path=f"{STUDY}/figures", labels=labels)


In [ ]:
fig = plot_helper.plot_jet_residuals()

In [ ]:
fig = plot_helper.plot_evt_res()

In [ ]:
pt_bins = np.array([0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200])

In [ ]:
fig = plot_helper.plot_jet_res_boxplot(bins=pt_bins)

In [ ]:
fig = plot_helper.plot_jet_response(pt_bins=pt_bins, use_energy=True)

In [ ]:
# Proxy jet-E IQR per 20 GeV bin of truth jet energy, the number studies/paper_tag_baseline compares.
names = [n["name"] for n in config_dict["networks"]] + ["pandora"]
print("| E [GeV] | " + " | ".join(f"{lo}-{hi}" for lo, hi in zip(pt_bins[:-1], pt_bins[1:])) + " |")
print("|---|" + "---|" * (len(pt_bins) - 1))
for name in names:
    if name not in perf_obj.data or "jet_residuals" not in perf_obj.data[name]:
        continue
    res = perf_obj.data[name]["jet_residuals"]
    row = []
    for lo, hi in zip(pt_bins[:-1], pt_bins[1:]):
        m = (res["ref_e"] > lo) & (res["ref_e"] < hi)
        v = res["e_rel"][m]
        row.append(f"{np.percentile(v, 75) - np.percentile(v, 25):.4f}" if len(v) else "-")
    print(f"| {name} | " + " | ".join(row) + " |")


In [ ]:
perf_obj.hung_match_particles(flatten=True, return_unmatched=True)

In [ ]:
qs = {"Charged": {"pt": 90, "eta": 80, "phi": 80}, "Neutral": {"pt": 90, "eta": 80, "phi": 80}}
fig = plot_helper.plot_residuals(pt_relative=True, log_y=True, qs=qs)

In [ ]:
qs = {"Neutral hadron": {"pt": 98, "eta": 75, "phi": 75}, "Photon": {"pt": 99, "eta": 90, "phi": 90}}
fig = plot_helper.plot_residuals_neutrals(pt_relative=True, log_y=True, qs=qs)

In [ ]:
eff_fr_colors = {
    "glow_paper_ref": {"neut had": "mediumseagreen", "photon": "tomato"},
    "glow_silu": {"neut had": "steelblue", "photon": "darkorange"},
    "glow_b200_jv": {"neut had": "crimson", "photon": "darkviolet"},
    "pandora": {"neut had": "dodgerblue", "photon": "goldenrod"},
}

In [ ]:
plot_helper.plot_eff_fr_purity(eff_fr_colors)

### Write every figure to `outputs_paper_tag_repro/`

In [ ]:
plot_helper.plot_event()
plot_helper.plot_particles()